# Intuition test: classification

Questions come first. For each one, write your prediction in the cell marked `Prediction`, then run the code cell below it. Answers are at the end, after every question. Do not scroll down early.

Covers notebooks 20 to 26. Regression questions are in `regression_intuition_test.ipynb`. Seeded and self-contained.

In [ ]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from common.datasets import load_breast_cancer_data, load_iris_flowers, make_credit_data

SEED = 42
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)

Xm, ym = make_moons(n_samples=400, noise=0.35, random_state=SEED)
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(Xm, ym, test_size=0.5, random_state=SEED)

## Q1. k = 1

You fit kNN on noisy two-class data (two half-moons with a lot of noise) with $k=1$, then with $k=25$. What is the training accuracy of the $k=1$ model, and why? Which $k$ do you expect to do better on fresh data?

Prediction: *(write here)*

In [ ]:
for k in (1, 25):
    m = KNeighborsClassifier(k).fit(Xm_tr, ym_tr)
    print(f"k={k:>2}: train {m.score(Xm_tr, ym_tr):.3f}  test {m.score(Xm_te, ym_te):.3f}")

## Q2. 99% accuracy

A fraud model scores 99% accuracy on data where 1% of transactions are fraud. Is it good? What is the least you would ask for before believing it? What score does a model that never flags anything get?

Prediction: *(write here)*

In [ ]:
X, y = make_credit_data(n_samples=20000, default_rate=0.01, seed=SEED)
never = np.zeros(len(y), dtype=int)
print("never flag: accuracy", accuracy_score(y, never), " recall", recall_score(y, never))

## Q3. C and gamma

An RBF SVM on the moons data. Predict the training and test accuracy for four settings: (C=1, gamma=1), (C=1, gamma=500), (C=0.01, gamma=1), (C=1000, gamma=500). Which one underfits, which one overfits, and does a huge C rescue a huge gamma?

Prediction: *(write here)*

In [ ]:
for C, g in [(1, 1), (1, 500), (0.01, 1), (1000, 500)]:
    m = SVC(C=C, gamma=g).fit(Xm_tr, ym_tr)
    print(
        f"C={C:>6}, gamma={g:>3}: train {m.score(Xm_tr, ym_tr):.3f}  test {m.score(Xm_te, ym_te):.3f}  support vectors {m.n_support_.sum()}"
    )

## Q4. One tree, many trees

You fit one unrestricted decision tree and one random forest of 200 trees on the same noisy data. What is each one's training accuracy? Which has the higher test accuracy, and by roughly how much?

Prediction: *(write here)*

In [ ]:
tree = DecisionTreeClassifier(random_state=SEED).fit(Xm_tr, ym_tr)
rf = RandomForestClassifier(200, random_state=SEED).fit(Xm_tr, ym_tr)
for name, m in [("tree", tree), ("forest", rf)]:
    print(f"{name:>6}: train {m.score(Xm_tr, ym_tr):.3f}  test {m.score(Xm_te, ym_te):.3f}")

## Q5. Which threshold

A loan model has calibrated probabilities. A missed default costs 10 times as much as a wrongly rejected good customer. Without computing anything else, what threshold on the default probability minimises expected cost? Then check against the data: where does the lowest total cost fall, and what does it do to recall?

Prediction: *(write here)*

In [ ]:
Xc, yc = make_credit_data(n_samples=20000, default_rate=0.05, seed=SEED)
Xc = Xc.fillna(Xc.median())
a, b, c, d = train_test_split(Xc, yc, test_size=0.5, random_state=SEED, stratify=yc)
p = (
    make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    .fit(a, c)
    .predict_proba(b)[:, 1]
)
d = d.to_numpy()
for t in (0.5, 0.2, 1 / 11, 0.05, 0.02):
    flag = p >= t
    cost = np.sum(flag & (d == 0)) * 1 + np.sum(~flag & (d == 1)) * 10
    print(
        f"threshold {t:.3f}: cost {cost:>5}  recall {recall_score(d, flag):.2f}  flagged {flag.mean():.2f}"
    )

## Q6. Who needs scaling

Compare each model on breast cancer, once on the raw features and once on standardised features. Which of kNN, SVM and a decision tree change a lot, and which do not move at all?

Prediction: *(write here)*

In [ ]:
Xb, yb = load_breast_cancer_data()
for name, m in [
    ("kNN", KNeighborsClassifier(5)),
    ("SVM", SVC()),
    ("tree", DecisionTreeClassifier(random_state=SEED)),
]:
    raw = cross_val_score(m, Xb, yb, cv=5).mean()
    sc = cross_val_score(make_pipeline(StandardScaler(), m), Xb, yb, cv=5).mean()
    print(f"{name:>5}: raw {raw:.3f}  scaled {sc:.3f}")

## Q7. Perfect separation

On iris, setosa is perfectly separable from the other species. You fit logistic regression with almost no penalty (C very large) and look at the weights. What do they do as C grows, and what does that mean for the predicted probabilities?

Prediction: *(write here)*

In [ ]:
Xi, yi = load_iris_flowers()
xs, ys = Xi[["petal_length", "petal_width"]].to_numpy(), (yi == 0).astype(int).to_numpy()
for C in (0.1, 10, 1e3, 1e6):
    m = LogisticRegression(C=C, max_iter=100000).fit(xs, ys)
    pr = m.predict_proba(xs)[:, 1]
    print(
        f"C={C:>9.1f}: weight norm {np.linalg.norm(m.coef_):6.2f}  most uncertain point p = {np.abs(pr - 0.5).min() + 0.5:.4f}"
    )

## Q8. Bagging against boosting on weak learners

Both methods get 200 decision stumps (trees of depth 1), each too simple to fit the moons well alone. Predict: does bagging 200 stumps help much? Does boosting 200 stumps? Why do they behave differently?

Prediction: *(write here)*

In [ ]:
stump = DecisionTreeClassifier(max_depth=1, random_state=SEED)
for name, m in [
    ("one stump", stump),
    ("bagged stumps", BaggingClassifier(stump, n_estimators=200, random_state=SEED)),
    ("boosted stumps", AdaBoostClassifier(stump, n_estimators=200, random_state=SEED)),
]:
    m.fit(Xm_tr, ym_tr)
    print(f"{name:>14}: train {m.score(Xm_tr, ym_tr):.3f}  test {m.score(Xm_te, ym_te):.3f}")

---

## Answers

**Q1.** Training accuracy for $k=1$ is exactly 1.000, because every training point is its own nearest neighbour. Test accuracy is 0.795, well below $k=25$ at 0.870. The $k=1$ model copies the noise, and $k=25$ averages it out, at the price of a training score of only 0.875. It is the same overfitting story as polynomial degree, with $k$ as the dial.

**Q2.** Not evidence of anything. A model that never flags a transaction scores 99% and has recall 0. Ask for recall and precision on the fraud class (or the PR curve), and compare against the always-negative baseline. 99% accuracy has to be set against the 99% you get for free.

**Q3.** (C=1, gamma=1) is the sensible one: train 0.88, test 0.855. (C=1, gamma=500) overfits: train 1.000, test 0.605, with almost every point a support vector. (C=0.01, gamma=1) underfits so heavily that it is close to chance on both sets (0.51 and 0.49). A huge $C$ does not rescue a huge gamma. (C=1000, gamma=500) is still at 1.000 and 0.615, because $C$ only makes the memorising stricter. The fix for a large gamma is a smaller gamma.

**Q4.** Both reach 1.000 on training data, so training accuracy says nothing here. On test data the single tree gets 0.810 and the forest 0.855, about 4.5 points better. Each deep tree is high variance, and averaging 200 differently resampled ones cancels much of it, while the bias of each tree is unchanged.

**Q5.** With calibrated probabilities the threshold is $c_{FP}/(c_{FP}+c_{FN})=1/11\approx0.09$. In the data, the lowest cost (3,575) is at 0.091, against 4,646 at 0.5, 4,178 at 0.05 and 6,684 at 0.02. Recall at the best threshold is 0.46 while flagging 12% of applicants. Going lower buys more recall (0.91 at 0.02) but the false alarms cost more than the extra catches save.

**Q6.** kNN and SVM change a lot: kNN goes from 0.928 to 0.965 and the SVM from 0.912 to 0.974. Both use distances, so a large-range feature dominates unless scaled. The tree is unchanged (0.917 both ways) because it only compares each feature with a threshold, and rescaling keeps the order of the values. Trees do not need scaling.

**Q7.** The weight norm keeps growing with $C$ (1.5, 4.5, 9.8, 12.7), because on separable data a larger weight always lowers the loss and no finite optimum exists. Even the least confident point is pushed to a probability of 0.998 at the top, so the probabilities are extreme and overconfident. The penalty is what keeps the weights finite, so keep it on.

**Q8.** Bagging 200 stumps does nothing: 0.755 test, identical to one stump. Every stump is equally too simple, so averaging identical errors keeps the error. Bagging reduces variance, and a stump is a high-bias model. Boosting lifts the test accuracy to 0.845 (train 0.940), because each new stump focuses on the points the earlier ones got wrong and so builds a more flexible boundary. Boosting reduces bias, bagging reduces variance. The right pairing is shallow trees for boosting and deep trees for bagging.